In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 4.8 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://spending-chastise-bullion.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://spending-chastise-bullion.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

# 【最新校長的祕密】定義 Google 搜尋工具
# 啟用此工具後，Gemini 就擁有了「即時上網搜尋」的能力（Google Search Grounding）
# 這就是為什麼當你問「現任校長是誰」，它能越過知識庫限制，查到當下 2026 年最新校長的原
google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],# 將上網搜尋工具綁定到聊天室中
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學，20字以內")
print(result)

新竹明新科大，私立科技大學，產學合作。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長為呂明峯。他於2025年2月1日正式上任，為該校第11任校長。 呂明峯教授在校服務34年，擁有豐富的業界經驗和學術背景，並曾打造全台首座半導體封裝測試類產線，推動成立半導體學院。

在他的任期內，呂明峯校長提出了學校未來的四大發展方向，旨在將明新科技大學打造成新竹地方人才庫、推動桃竹苗大矽谷的引擎、新南向專班的基地，並活化資源以實現永續校園。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ua5551030dc64321324b89c4f0ea9257d","events":[{"type":"message","message":{"type":"text","id":"616420893933699294","quoteToken":"TfjoJgEF1HP7pgkXjutxlzW1Lq5sGeAqvFzihQo0i40gkc05-n3itAUsEw_3fL-VVuKbeGlxgWCwD0RNRfwd-kuTaqfHdYDh_4Si2x6v9ASTGJjMLy91isXzTlDfmIk8pP4TyWt40jtNHhEBMiRDiQ","markAsReadToken":"pNm2h_cyIiUpCYHLER_Xix3MFRe7I4dxML6yWfWYfb3Dq_2e5D8doU3kVP8tbeBbD4K9jR534eEQ18wpO1quifQFBQnzOKTtMPmHQ73Pow4rc-l-rKiXQ3Qe49ffl7egD_16ExUAT9txhjQpUMTePzWbhShJjc7ypSAzPIYHd0cJtbpgdswc-QWlHayea6feI5puZ_lEO1MlskdC9RwQlQ","text":"AI  簡介明新科技大學，20字以內"},"webhookEventId":"01KSZFMEJ33A65QT7DARYMDHG1","deliveryContext":{"isRedelivery":false},"timestamp":1780246788166,"source":{"type":"user","userId":"Ue65570520db252700cb187ff02c70937"},"replyToken":"3d7a5e419259417581c79a78da08e735","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [31/May/2026 16:59:51] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Ua5551030dc64321324b89c4f0ea9257d","events":[{"type":"message","message":{"type":"text","id":"616420901953208593","quoteToken":"yWcL2mCuuM1TKcYKq9xW5nwUn5yrY_7vmCvVGBpxT2lCaLo59Dq6gSU8oJV35RAKi-LeVwcoZEhWcdI9aeQlf40IpM7Qw_XLaJHP7eL42tYmhw8pmjzRAXQnhaQsZLOkltdG8j-aCdCcrh4iPeYohw","markAsReadToken":"q1Qp0E73Ky2pG7x8Rj72onDiXUgz6FyFI0wKCf0aCCXrhJem4r8NnYBATHhYQXX7c8tBNy2RMxFSxw7Z4f5KpG4mWn0PSTWp_0xuopVhA0sfSIgcV7TExXKCq02L0Ij0YP1dWCy_jPcSwdM4mmtVvFpQWAQ2tUb3etZLFDPySK5Ben7_Sf9vfyWLmQdXt5kBisBH1YD8yBJ39OpB8e_g1g","text":"AI  校長是誰"},"webhookEventId":"01KSZFMK3GZ1XV0G35Q4D41AC0","deliveryContext":{"isRedelivery":false},"timestamp":1780246792943,"source":{"type":"user","userId":"Ue65570520db252700cb187ff02c70937"},"replyToken":"cb6ee0ec59e543fcbb005d59bc6828cb","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [31/May/2026 16:59:55] "POST / HTTP/1.1" 200 -
